# 05 — Multimodal Feature Fusion (Transformer)
**AnemiaFusionNet — Stage 5**

Combines the three modality embeddings — **image**, **clinical**, **geo-risk** —
into one fused representation using a small Transformer encoder that treats each
modality embedding as one "token" in a length-3 (or 4 with a CLS token) sequence.

This design lets the self-attention layers learn **which modality to trust more per
patient** (e.g., down-weighting a blurry image, up-weighting geo-risk in
high-prevalence districts) instead of naive concatenation.

## 5.1 Set Working Directory and Load Packages

In [1]:
import os
from pathlib import Path

# Change working directory to project root if executed from notebooks folder
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
print("Current working directory:", os.getcwd())

Current working directory: c:\Users\ratha\OneDrive\Desktop\datavidwan\New folder (2)\files


## 5.2 Verify Modality Embeddings and Lookups
We verify that the visual and clinical embeddings exported in notebooks 03 and 04 exist and have matching dimensions.

In [2]:
import torch
import pandas as pd
import pickle

img_embs = torch.load("data/embeddings/image_embeddings.pt")
clin_embs = torch.load("data/embeddings/clinical_embeddings.pt")
master_df = pd.read_csv("data/processed/master_dataset.csv")

print("Image embeddings shape:", img_embs.shape)
print("Clinical embeddings shape:", clin_embs.shape)
print("Master dataset patients:", len(master_df))

assert img_embs.shape[0] == len(master_df), "Image embeddings patient count mismatch!"
assert clin_embs.shape[0] == len(master_df), "Clinical embeddings patient count mismatch!"

Image embeddings shape: torch.Size([217, 256])
Clinical embeddings shape: torch.Size([217, 256])
Master dataset patients: 217


## 5.3 Instantiate ModalityFusionTransformer
We load `ModalityFusionTransformer` from `src.fusion_transformer` and perform a forward pass on dummy inputs, testing the attention weights extraction.

In [3]:
from src.fusion_transformer import ModalityFusionTransformer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Running on device:", device)

fusion_model = ModalityFusionTransformer(d_model=256, n_heads=8, n_layers=4, dim_ff=512).to(device)

# Generate dummy embeddings: batch=4, dim=256
dummy_img = torch.randn(4, 256).to(device)
dummy_clin = torch.randn(4, 256).to(device)
dummy_geo = torch.randn(4, 256).to(device)

# Run forward pass with return_attn=True to extract attention maps
logits, attn_weights = fusion_model(dummy_img, dummy_clin, dummy_geo, return_attn=True)

print("Logits shape:", logits.shape)  # Should be (4, 2)
print("Attention weights shape:", attn_weights.shape)  # Should be (4, 4, 4)

assert logits.shape == (4, 2), "Logits shape mismatch!"
assert attn_weights.shape == (4, 4, 4), "Attention weights shape mismatch!"

Running on device: cuda
Logits shape: torch.Size([4, 2])
Attention weights shape: torch.Size([4, 4, 4])


## 5.4 Verify AnemiaFusionNet Full Wrapper
We construct the complete `AnemiaFusionNet` wrapper combining the image encoder, clinical encoder, and geo risk encoder with the fusion transformer.

In [4]:
from src.image_encoder import ImageEncoder
from src.clinical_encoder import ClinicalEncoder
from src.geo_encoder import GeoRiskEncoder
from src.fusion_transformer import AnemiaFusionNet

# Instantiate encoders
image_encoder = ImageEncoder(embed_dim=256, pretrained=False)
clinical_encoder = ClinicalEncoder(num_numeric=13, cat_cardinalities=[2, 2, 2], embed_dim=256)
geo_encoder = GeoRiskEncoder(num_states=36, state_emb_dim=16, d_model=256)

# Complete wrapper
fusion_net = AnemiaFusionNet(image_encoder, clinical_encoder, geo_encoder, d_model=256).to(device)

# Dummy raw inputs
dummy_image = torch.randn(2, 3, 224, 224).to(device)
dummy_x_num = torch.randn(2, 13).to(device)
dummy_x_cat = torch.zeros(2, 3, dtype=torch.long).to(device)
dummy_state = torch.zeros(2, dtype=torch.long).to(device)
dummy_risk = torch.rand(2, 1).to(device)

logits = fusion_net(dummy_image, dummy_x_num, dummy_x_cat, dummy_state, dummy_risk)
print("Wrapper logits shape:", logits.shape)  # Should be (2, 2)
assert logits.shape == (2, 2), "Wrapper logits shape mismatch!"
print("AnemiaFusionNet full wrapper forward pass succeeded on GPU/device!")

Wrapper logits shape: torch.Size([2, 2])
AnemiaFusionNet full wrapper forward pass succeeded on GPU/device!
